# Clasificación Binaria: Regresión Logística


**Autores:**
- Miguel David Dávila Palomino
- Diego Santiago Yáñez Chile
- María Fernanda Zárate Prado

*Redes:* [@neuropucp](https://www.instagram.com/neuropucp)

---

## Predicción de Ataques de Pánico a partir de Señales Fisiológicas

---

Bienvenido/a a este tutorial interactivo sobre **Regresión Logística**.
Este tutorial está basado en la [Machine Learning Specialization de Andrew Ng](https://www.coursera.org/specializations/machine-learning-introduction)
(disponible también en el canal de YouTube de [DeepLearning.AI](https://www.youtube.com/@deeplearningai)).

### 🎯 Objetivos de Aprendizaje

Al finalizar este tutorial serás capaz de:

1. ❌ Explicar **por qué la regresión lineal falla estructuralmente** en tareas de clasificación binaria.
2. ✨ Derivar e interpretar la **función sigmoide** como estimador de probabilidades.
3. 🔗 Conectar la formulación de ML con el marco clásico de **log-odds (logit)** de estadística — demostrando que son el mismo modelo (¡con demostración algebraica incluida!).
4. 📐 Recorrer la **derivación paso a paso** de cómo la Estimación por Máxima Verosimilitud (MLE) genera la pérdida de entropía cruzada binaria.
5. 🎚️ Comprender el papel de la **tasa de aprendizaje** $\alpha$ en el descenso de gradiente.
6. 🎬 Usar widgets interactivos para **ver aprender al modelo** parámetro a parámetro.

---

### Escenario Clínico

Imagina que eres investigador/a clínico/a en una clínica de trastornos de sueño y ansiedad.
Has recolectado datos fisiológicos de **120 participantes** durante una prueba de estrés estándar.
Antes de la prueba registraste:

| Variable | Descripción |
|---|---|
| **Frecuencia Cardíaca (BPM)** | Medida con un sensor de muñeca de grado médico |
| **Nivel de Estrés Autorreportado** | Puntuación del 1 (tranquilo) al 10 (máximo estrés) |

Para cada participante registraste un resultado clínico binario:

> **¿El participante experimentó un ataque de pánico durante la prueba?**
> - `y = 1` → **Sí**, ocurrió un ataque de pánico.
> - `y = 0` → **No**, no ocurrió ataque de pánico.

Tu objetivo: construir un modelo que prediga la **probabilidad** de un ataque de pánico dados los registros pre-prueba de un nuevo paciente. Esto es un problema de **clasificación binaria**.


In [ ]:
# @title Configuración — Ejecuta esta celda primero (oculta en la vista del libro)
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from mpl_toolkits.mplot3d import Axes3D
from ipywidgets import interact, widgets
from IPython.display import display
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings("ignore")

RNG = np.random.default_rng(42)

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "#f8f9fa",
    "axes.grid":        True,
    "grid.color":       "white",
    "grid.linewidth":   1.2,
    "font.size":        12,
    "axes.titlesize":   14,
    "axes.labelsize":   12,
})
BLUE   = "#2196F3"
RED    = "#F44336"
GREEN  = "#4CAF50"
ORANGE = "#FF9800"
PURPLE = "#9C27B0"

# ═══════════════════════════════════════════════════════════════════════
# GENERACIÓN DE DATOS
# ═══════════════════════════════════════════════════════════════════════

# Datos 1-D: separación obvia en 80 BPM (Widgets 1, 2 y 3)
hr_1d_0 = RNG.uniform(55, 79, 60)
hr_1d_1 = RNG.uniform(81, 125, 60)
HR1D    = np.concatenate([hr_1d_0, hr_1d_1])
Y1D     = np.concatenate([np.zeros(60), np.ones(60)])
jitter_1d = RNG.uniform(-0.025, 0.025, len(Y1D))

HR1D_mean, HR1D_std = HR1D.mean(), HR1D.std()
HR1D_s = (HR1D - HR1D_mean) / HR1D_std

# Regresión lineal 1-D (para Widget 1)
LIN_REG_1D = LinearRegression().fit(HR1D.reshape(-1,1), Y1D)

# ═══════════════════════════════════════════════════════════════════════
# FUNCIONES AUXILIARES
# ═══════════════════════════════════════════════════════════════════════

def sigmoid_fn(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

def logistic_loss(w, b, X, y):
    p = np.clip(sigmoid_fn(X * w + b), 1e-10, 1 - 1e-10)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

def accuracy_fn(w, b, X, y):
    return np.mean((sigmoid_fn(X * w + b) >= 0.5).astype(float) == y)

# ═══════════════════════════════════════════════════════════════════════
# DESCENSO DE GRADIENTE PRE-COMPUTADO (Widget 3)
# ═══════════════════════════════════════════════════════════════════════

N_ITER   = 150
LR_GD    = 0.5
w_hist   = np.zeros(N_ITER + 1)
b_hist   = np.zeros(N_ITER + 1)
J_hist   = np.zeros(N_ITER + 1)
acc_hist = np.zeros(N_ITER + 1)

w_hist[0]   = -2.0
b_hist[0]   =  1.0
J_hist[0]   = logistic_loss(w_hist[0], b_hist[0], HR1D_s, Y1D)
acc_hist[0] = accuracy_fn(w_hist[0], b_hist[0], HR1D_s, Y1D)

for _t in range(N_ITER):
    _z    = HR1D_s * w_hist[_t] + b_hist[_t]
    _yhat = sigmoid_fn(_z)
    _dw   = np.mean((_yhat - Y1D) * HR1D_s)
    _db   = np.mean(_yhat - Y1D)
    w_hist[_t+1]   = w_hist[_t] - LR_GD * _dw
    b_hist[_t+1]   = b_hist[_t] - LR_GD * _db
    J_hist[_t+1]   = logistic_loss(w_hist[_t+1], b_hist[_t+1], HR1D_s, Y1D)
    acc_hist[_t+1] = accuracy_fn(w_hist[_t+1], b_hist[_t+1], HR1D_s, Y1D)

# Cuadrícula de contorno para Widget 3
w_range = np.linspace(-3, 6, 150)
b_range = np.linspace(-4, 4, 150)
J_GRID  = np.array([
    [logistic_loss(w, b, HR1D_s, Y1D) for w in w_range]
    for b in b_range
])

# ═══════════════════════════════════════════════════════════════════════
# SUPERFICIES 3-D PARA WIDGET 2
# ═══════════════════════════════════════════════════════════════════════

w_surf_r = np.linspace(-4, 8, 60)
b_surf_r = np.linspace(-5, 5, 60)
W_SURF, B_SURF = np.meshgrid(w_surf_r, b_surf_r)
J_LOG_SURF = np.zeros_like(W_SURF)
J_SQ_SURF  = np.zeros_like(W_SURF)
for _i in range(W_SURF.shape[0]):
    for _j in range(W_SURF.shape[1]):
        _w, _b = W_SURF[_i,_j], B_SURF[_i,_j]
        _z2 = HR1D_s * _w + _b
        _ph = np.clip(sigmoid_fn(_z2), 1e-10, 1-1e-10)
        J_LOG_SURF[_i,_j] = -np.mean(Y1D*np.log(_ph)+(1-Y1D)*np.log(1-_ph))
        J_SQ_SURF[_i,_j]  =  np.mean((sigmoid_fn(_z2) - Y1D)**2)

def _gd_path(mode, n=55, lr=0.3):
    w, b = -3.0, 3.0
    ws, bs = [w], [b]
    for _ in range(n):
        z = HR1D_s * w + b
        yh = sigmoid_fn(z)
        if mode == "log":
            dw = np.mean((yh - Y1D) * HR1D_s)
            db = np.mean(yh - Y1D)
        else:
            dw = np.mean(2*(yh-Y1D)*yh*(1-yh)*HR1D_s)
            db = np.mean(2*(yh-Y1D)*yh*(1-yh))
        w -= lr*dw; b -= lr*db
        ws.append(w); bs.append(b)
    return np.array(ws), np.array(bs)

WS_LOG, BS_LOG = _gd_path("log")
WS_SQ,  BS_SQ  = _gd_path("sq")

def _cost_at(w, b, mode):
    z  = HR1D_s * w + b
    ph = np.clip(sigmoid_fn(z), 1e-10, 1-1e-10)
    if mode == "log":
        return -np.mean(Y1D*np.log(ph)+(1-Y1D)*np.log(1-ph))
    return np.mean((sigmoid_fn(z)-Y1D)**2)

print("Configuracion completa. Todos los widgets estan listos.")


Configuracion completa. Todos los widgets estan listos.


---
## 🧠 Sección 1: El Problema de Clasificación — ¿Por qué no simplemente usar Regresión Lineal?

Esta es la primera pregunta natural, y merece una respuesta cuidadosa.

Ya conoces cómo modelar un resultado continuo con **Regresión Lineal**:

$$f_{\vec{w},b}(\vec{x}) = \vec{w} \cdot \vec{x} + b$$

En nuestro contexto clínico, aplicada a predecir ataques de pánico:

$$\hat{y} = w_1 \cdot \text{FrecuenciaCardíaca} + b$$

Este modelo inevitablemente produce valores como $1.8$ (para un paciente muy ansioso) o $-0.3$ (para uno muy tranquilo). ¿Qué significa una *probabilidad del 180%*? Es matemáticamente indefinido y clínicamente inútil. ❌

Más profundamente, un ajuste de mínimos cuadrados sobre datos binarios es distorsionado desproporcionadamente por los puntos extremos. Este no es un problema de ajuste — es una **falla estructural**.

**Lo que realmente necesitamos es una función que:**
1. Tome como entrada cualquier combinación lineal real (de $-\infty$ a $+\infty$).
2. La mapee a una probabilidad válida — un número estrictamente entre **0** y **1**.
3. Produzca una curva en S, coherente con la intuición de que el riesgo *se acelera* alrededor de un umbral y luego *se satura* en los extremos.

---

### ✨ La Solución: La Función Sigmoide (Logística)

La **función sigmoide** hace exactamente esto:

$$g(z) = \frac{1}{1 + e^{-z}}$$

Piensa en $z$ como el "puntaje en bruto" del modelo lineal ($\vec{w} \cdot \vec{x} + b$). La sigmoide comprime este puntaje sin límites al intervalo $(0, 1)$:

| Puntaje bruto $z$ | Intuición | Salida sigmoide $g(z)$ |
|---|---|---|
| Muy negativo (e.g., $-8$) | 🔵 Evidencia fuerte de NO ataque de pánico | $\approx 0.0003$ |
| Alrededor de $-2$ | 🔹 Evidencia moderada de no pánico | $\approx 0.12$ |
| Exactamente $0$ | ⚖️ Máxima incertidumbre | $= 0.50$ |
| Alrededor de $+2$ | 🔸 Evidencia moderada de pánico | $\approx 0.88$ |
| Muy positivo (e.g., $+8$) | 🔴 Evidencia fuerte de ataque de pánico | $\approx 0.9997$ |

Componemos la sigmoide con el modelo lineal para formar la **Regresión Logística**:

$$\boxed{f_{\vec{w},b}(\vec{x}) = g\!\left(\vec{w} \cdot \vec{x} + b\right) = \frac{1}{1 + e^{-(\vec{w} \cdot \vec{x} + b)}}}$$

La salida es la estimación calibrada del modelo de $P(\text{ataque de pánico} = 1 \mid \vec{x})$. ✅

---

> ### 🔗 Conectando Machine Learning con tu Clase de Estadística: El Logit y los Log-Odds
>
> Si has tomado una clase estándar de estadística, quizás hayas visto la Regresión Logística escrita de manera diferente: la ecuación de **"Logit"** o **"Log de Momios (Log-Odds)"**:
>
> $$\ln\!\left(\frac{p}{1-p}\right) = \vec{w} \cdot \vec{x} + b$$
>
> Donde $p$ es la probabilidad del evento (e.g., un ataque de pánico).
>
> #### ⚠️ ¿Por qué no podemos usar la salida sigmoide $\hat{p}$ directamente como variable de respuesta en una regresión lineal?
>
> La salida $\hat{p}$ está **acotada entre 0 y 1** — y eso es exactamente el problema. La regresión lineal ordinaria asume que la variable de respuesta puede tomar cualquier valor en $(-\infty, +\infty)$. Usar $\hat{p}$ viola esa suposición de dos formas:
>
> 1. **La relación NO es lineal** 📐: Entre $\vec{x}$ y $\hat{p}$ existe la curva en S, no una línea recta. Forzar un modelo lineal sobre una curva produce predicciones fuera de $[0,1]$ en los extremos (como verás en el Widget 1 👇).
> 2. **El espacio de valores es incorrecto** ❌: Las probabilidades viven en $(0,1)$; la regresión lineal las predice fuera de ese rango de forma inevitable.
>
> La transformación **log-odds** resuelve ambos problemas de una sola vez: $\ln\!\left(\frac{p}{1-p}\right)$ va de $-\infty$ a $+\infty$ y mantiene una relación **perfectamente lineal** con $\vec{w} \cdot \vec{x} + b$. ✅
>
> ---
>
> #### 📐 Demostración Algebraica: ¿Por qué $e^z$ son las Odds?
>
> Esta es una de las demostraciones más satisfactorias del machine learning. No *suponemos* que $e^z$ son las Odds — es una **certeza matemática directa** que se obtiene despejando la función Sigmoide. Todo lo que haremos es tomar la fórmula de la Sigmoide y resolver para $e^z$.
>
> **El punto de partida:** definamos los términos.
> - **Probabilidad ($p$):** la salida de la función Sigmoide.
> - **Odds:** por definición en estadística, son la Probabilidad de Éxito dividida entre la Probabilidad de Fracaso: $\frac{p}{1-p}$.
>
> Vamos a probar que $e^z = \frac{p}{1-p}$.
>
> **Paso 1:** Empezamos con la función Sigmoide estándar.
> $$p = \frac{1}{1 + e^{-z}}$$
>
> **Paso 2:** Multiplicamos ambos lados por el denominador (sacamos $e^{-z}$ del denominador).
> $$p\,(1 + e^{-z}) = 1$$
>
> **Paso 3:** Dividimos ambos lados entre $p$.
> $$1 + e^{-z} = \frac{1}{p}$$
>
> **Paso 4:** Restamos 1 de ambos lados.
> $$e^{-z} = \frac{1}{p} - 1$$
>
> **Paso 5:** Buscamos denominador común (reescribimos $1$ como $\frac{p}{p}$).
> $$e^{-z} = \frac{1}{p} - \frac{p}{p} = \frac{1-p}{p}$$
>
> **Paso 6:** Tomamos el recíproco de ambos lados. Recordemos que $e^{-z} = \frac{1}{e^z}$, así que invertir ambos lados convierte $e^{-z}$ en $e^z$:
> $$\boxed{e^z = \frac{p}{1-p}}$$
>
> 🎯 **¡El momento "Ajá"!** El lado derecho de esa ecuación, $\frac{p}{1-p}$, es exactamente la definición estadística de las **Odds**. Por álgebra pura hemos demostrado que:
> $$e^z = \text{Odds}$$
>
> Y tomando el logaritmo natural de ambos lados para bajar el $z$:
> $$z = \ln(\text{Odds}) = \text{Log-Odds}$$
>
> El mundo de ML usa $z = \vec{w}\cdot\vec{x}+b$ porque quiere trazar una línea recta. El mundo de estadística usa Log-Odds porque quiere analizar riesgo. Este trozo de álgebra demuestra que están haciendo **literalmente lo mismo**, solo abordándolo desde lados opuestos del signo de igualdad. 🤯
>
> ---
>
> #### 🔄 El Círculo Completo: De Odds de vuelta a Probabilidad
>
> Ahora que sabemos que $e^z = \text{Odds}$, podemos también ver **todas las formas equivalentes** de escribir la ecuación de probabilidad:
>
> $$\hat{p} = \frac{\text{Odds}}{1 + \text{Odds}} = \frac{e^z}{1+e^z} = \frac{1}{1+e^{-z}} = \sigma(z)$$
>
> Estas cuatro expresiones son algebraicamente idénticas. La primera (con Odds) es la más fácil de derivar; la última (función Sigmoide) es la más compacta para implementar.
>
> ---
>
> #### 📊 Ejemplo numérico
>
> Observa cómo el score lineal $z$, la probabilidad $\hat{p}$ y el log-odds se relacionan:
>
> | Score lineal $z$ | Probabilidad $\hat{p} = \sigma(z)$ | Odds $= e^z$ | Log-Odds $= \ln(\text{Odds})$ |
> |:---:|:---:|:---:|:---:|
> | $-3$ | $0.047$ | $0.050$ | $-3.0$ |
> | $-1$ | $0.269$ | $0.368$ | $-1.0$ |
> | $\phantom{-}0$ | $0.500$ | $1.000$ | $\phantom{-}0.0$ |
> | $+1$ | $0.731$ | $2.718$ | $+1.0$ |
> | $+3$ | $0.953$ | $20.09$ | $+3.0$ |
>
> 🔑 **¡La columna de Log-Odds es exactamente igual a $z$!** Esto no es coincidencia — es la definición: $\text{Log-Odds} = z = \vec{w} \cdot \vec{x} + b$.
>
> La transformación logit convierte la curva S acotada en una escala perfectamente lineal sin restricciones. En otras palabras: **estamos haciendo regresión lineal sobre el log-odds** — no sobre la probabilidad directamente.
>
> #### 🤖 ¿Cómo obtenemos el Log-Odds a partir de un modelo ML ya entrenado?
>
> Una vez entrenado el modelo (obtenidos $\vec{w}$ y $b$), el log-odds para cualquier paciente es simplemente la **combinación lineal antes de aplicar la sigmoide**:
>
> $$\underbrace{\vec{x}}_{\text{Características}} \;\xrightarrow{\vec{w},\,b}\; \underbrace{z = \vec{w}\cdot\vec{x}+b}_{\text{Log-Odds}} \;\xrightarrow{\sigma(\cdot)}\; \underbrace{\hat{p} = \frac{e^z}{1+e^z}}_{\text{Probabilidad}} \;\xrightarrow{\geq 0.5}\; \underbrace{\hat{y}}_{\text{Clasificación}}$$
>
> **Ejemplo concreto** 🏥: Supongamos que tras el entrenamiento obtenemos $w_1 = 0.15$ (peso para FC) y $b = -12.0$:
>
> | Paciente | FC (BPM) | $z = 0.15 \times \text{FC} - 12$ | Odds $= e^z$ | $\hat{p} = \frac{e^z}{1+e^z}$ | Clasificación |
> |---|:---:|:---:|:---:|:---:|:---:|
> | A (baja FC) | 65 | $-2.25$ | $0.105$ | $9.5\%$ | ✅ No pánico |
> | B (FC límite) | 80 | $\phantom{-}0.00$ | $1.000$ | $50.0\%$ | ⚖️ Límite |
> | C (alta FC) | 100 | $+3.00$ | $20.09$ | $95.2\%$ | ⚠️ Pánico |
>
> **Interpretación del peso** 💡: $w_1 = 0.15$ significa que cada BPM adicional de frecuencia cardíaca multiplica las *odds* de un ataque de pánico por $e^{0.15} \approx 1.16$ — un aumento del **16% en los momios** por BPM.



### 📊 Demo Interactiva 1: La Frontera de Decisión en 1D

**Tu tarea 👇:** Activa y desactiva cada modelo con los botones de palanca.

**Qué buscar 🔍:**
1. **Solo "Regresión Lineal"** 🟠: La línea naranja sale del rango válido $[0, 1]$ — predice probabilidades negativas para FC baja y mayores a 1 para FC alta. Las zonas sombreadas marcan estas regiones inválidas. ❌
2. **Solo "Regresión Logística"** 🟣: La curva en S se ajusta perfectamente al escalón en ~80 BPM, permaneciendo siempre dentro de $(0, 1)$. ✅
3. **Ambas activas** ⚖️: Compara cómo la curva en S captura la transición abrupta que la línea recta no puede modelar sin salirse del rango.

**🎯 Conclusión**: La curva en S no es solo una preferencia estética — es la única forma matemáticamente correcta de modelar una probabilidad con una transición abrupta.


In [ ]:
def plot_1d_boundary(mostrar_lineal=False, mostrar_logistica=True):
    fig, ax = plt.subplots(figsize=(10, 6))

    ax.scatter(HR1D[Y1D==0], Y1D[Y1D==0] + jitter_1d[Y1D==0],
               c=BLUE, alpha=0.70, s=65, edgecolors="white", lw=0.7,
               label="Sin ataque de panico (y=0)", zorder=3)
    ax.scatter(HR1D[Y1D==1], Y1D[Y1D==1] + jitter_1d[Y1D==1],
               c=RED,  alpha=0.70, s=65, edgecolors="white", lw=0.7,
               label="Ataque de panico (y=1)", zorder=3)

    hr_plot   = np.linspace(48, 135, 600)
    hr_plot_s = (hr_plot - HR1D_mean) / HR1D_std

    if mostrar_lineal:
        y_lin = LIN_REG_1D.predict(hr_plot.reshape(-1, 1))
        ax.plot(hr_plot, y_lin, color=ORANGE, lw=2.5, ls="-.",
                label="Regresion Lineal", zorder=4)
        invalid_low  = y_lin < 0
        invalid_high = y_lin > 1
        if invalid_low.any():
            ax.fill_between(hr_plot, y_lin, 0,
                            where=invalid_low, color="#FDD835",
                            alpha=0.35, zorder=2)
        if invalid_high.any():
            ax.fill_between(hr_plot, y_lin, 1,
                            where=invalid_high, color="#66BB6A",
                            alpha=0.35, zorder=2)
        ax.annotate("P < 0  (region invalida)",
                    xy=(55, -0.10), fontsize=10, color="#7B6000",
                    bbox=dict(boxstyle="round,pad=0.3",
                              fc="#fff9c4", alpha=0.9))
        ax.annotate("P > 1  (region invalida)",
                    xy=(105, 1.08), fontsize=10, color="#1B5E20",
                    bbox=dict(boxstyle="round,pad=0.3",
                              fc="#c8f5c8", alpha=0.9))

    if mostrar_logistica:
        p_hat = sigmoid_fn(w_hist[-1] * hr_plot_s + b_hist[-1])
        ax.plot(hr_plot, p_hat, color=PURPLE, lw=3,
                label="Regresion Logistica (S-curve)", zorder=5)
        ax.fill_between(hr_plot, 0, p_hat, alpha=0.08, color=PURPLE)

    ax.axhline(0.5, color="gray", lw=1.2, ls=":", alpha=0.65)
    ax.axvline(80,  color="dimgray", lw=1.8, ls="--", alpha=0.55,
               label="80 BPM (umbral real)")
    ax.axhspan(1.0, 1.30, color="#c8f5c8", alpha=0.15)
    ax.axhspan(-0.30, 0.0, color="#fff9c4", alpha=0.20)
    ax.set_xlim(48, 135)
    ax.set_ylim(-0.30, 1.30)
    ax.set_xlabel("Frecuencia Cardiaca (BPM)", fontweight="bold")
    ax.set_ylabel("P(Ataque de Panico)", fontweight="bold")
    ax.set_title("Frontera de Decision: Regresion Lineal vs. Logistica",
                 fontweight="bold")
    ax.legend(loc="center left", framealpha=0.9)
    plt.tight_layout()
    plt.show()

interact(
    plot_1d_boundary,
    mostrar_lineal    = widgets.ToggleButton(
        value=False, description="Regresion Lineal",
        button_style="warning",
        layout=widgets.Layout(width="180px", height="36px")),
    mostrar_logistica = widgets.ToggleButton(
        value=True,  description="Regresion Logistica",
        button_style="info",
        layout=widgets.Layout(width="190px", height="36px")),
)


interactive(children=(ToggleButton(value=False, button_style='warning', description='Regresion Lineal', layout…

<function __main__.plot_1d_boundary(mostrar_lineal=False, mostrar_logistica=True)>

---
## 💰 Sección 2: La Función de Costo — ¿Cómo Sabe el Modelo Cuándo se Equivoca?

Tenemos una arquitectura de modelo. Ahora necesitamos una forma de medir **qué tan equivocado** está, para poder mejorarlo. Esta medición es la **función de costo** (o **función de pérdida**).

Para la regresión logística, la elección correcta es la **Entropía Cruzada Binaria** (también llamada **Log Loss** o **Pérdida Logarítmica**). ¿Por qué esta función específica y no, digamos, el Error Cuadrático Medio? La respuesta viene de un principio fundamental de la estadística: la **Estimación por Máxima Verosimilitud (MLE)**.

---

> ### Paso a Paso: De MLE a la Pérdida Logística
>
> En estadística, MLE es un método para encontrar los parámetros del modelo ($\vec{w}$, $b$) que hacen que los **datos observados sean los más probables**. Derivemos la pérdida logística desde cero.
>
> ---
>
> **Paso 1 — Probabilidad de una sola predicción**
>
> Sea $\hat{y} = f_{\vec{w},b}(\vec{x})$ la salida del modelo (una probabilidad). La etiqueta verdadera $y$ es 0 o 1. Podemos escribir *ambos* casos como una sola expresión de Bernoulli:
>
> $$P(y \mid x) = \hat{y}^{y} \cdot (1 - \hat{y})^{(1 - y)}$$
>
> ¿Por qué funciona?
> - Si $y = 1$: el segundo factor $(1-\hat{y})^0 = 1$, dejando solo $\hat{y}$.
> - Si $y = 0$: el primer factor $\hat{y}^0 = 1$, dejando solo $(1-\hat{y})$.
>
> ---
>
> **Paso 2 — Verosimilitud del conjunto de datos completo**
>
> Para un conjunto de $m$ ejemplos **independientes**, la Verosimilitud total es el producto de todas las probabilidades individuales:
>
> $$L = \prod_{i=1}^{m} \left( \hat{y}^{(i)} \right)^{y^{(i)}} \left( 1 - \hat{y}^{(i)} \right)^{(1 - y^{(i)})}$$
>
> El objetivo de MLE es **maximizar** $L$.
>
> ---
>
> **Paso 3 — El "Log" en Log-Verosimilitud**
>
> Multiplicar millones de probabilidades pequeñas (todas entre 0 y 1) produce números tan pequeños que una computadora los redondea a cero — un problema llamado **desbordamiento numérico negativo**. La solución: tomar el logaritmo natural. Dos reglas hacen la álgebra hermosa:
>
> 1. $\log(a \cdot b) = \log(a) + \log(b)$ — productos se convierten en sumas
> 2. $\log(a^b) = b \cdot \log(a)$ — los exponentes bajan al frente
>
> Aplicando estas reglas, el producto gigante se transforma en una suma limpia:
>
> $$\log(L) = \sum_{i=1}^{m} \left[ y^{(i)}\log(\hat{y}^{(i)}) + (1 - y^{(i)})\log(1 - \hat{y}^{(i)}) \right]$$
>
> ---
>
> **Paso 4 — Convertir maximización en minimización**
>
> El descenso de gradiente *minimiza*. Para convertir la maximización de $\log(L)$ en un problema de minimización, multiplicamos por $-1$. Dividimos por $m$ para promediar sobre el conjunto de datos:
>
> $$\boxed{J(\vec{w},b) = -\frac{1}{m} \sum_{i=1}^{m} \left[ y^{(i)}\log\!\left(f_{\vec{w},b}(\vec{x}^{(i)})\right) + (1 - y^{(i)})\log\!\left(1 - f_{\vec{w},b}(\vec{x}^{(i)})\right) \right]}$$
>
> Esta es la **Pérdida Logarítmica Negativa (NLL)**, también llamada **Entropía Cruzada Binaria**.
>
> **Conclusión**: Minimizar esta pérdida específica es matemáticamente idéntico a encontrar los parámetros que maximizan la probabilidad de observar los datos de entrenamiento.

---

### ¿Por qué la Forma de la Superficie de Costo Importa?

La forma de $J$ como función de los parámetros determina si el descenso de gradiente puede encontrar el mejor modelo de forma confiable:

| Función de Pérdida | Forma de la Superficie | Resultado del Descenso de Gradiente |
|---|---|---|
| **Error Cuadrático** (aplicado a salidas logísticas) | No convexa — múltiples mínimos locales | Puede converger a un modelo subóptimo |
| **Entropía Cruzada Binaria / NLL** | **Convexa** — un único cuenco suave | **Garantiza** la convergencia al mínimo global |

Esta convexidad no es coincidencia — es consecuencia matemática directa de usar MLE con un modelo de familia exponencial (distribución de Bernoulli). Nos da una **garantía clínica** de que el modelo entrenado es verdaderamente óptimo.


### 🏔️ Demo Interactiva 2: El Paisaje de la Función de Costo en 3D

La función de costo $J(\vec{w}, b)$ es una superficie en el espacio de parámetros. Aquí la visualizamos en función del peso $w$ (qué tanto influye la FC en las predicciones) y el sesgo $b$.

Ambas superficies se muestran **simultáneamente** para facilitar la comparación directa:
- **Izquierda 🟢 — Pérdida Logística (NLL)**: cuenco convexo con un único mínimo global.
- **Derecha 🔴 — Error Cuadrático**: superficie no convexa con múltiples valles locales.

**Qué hacer 👇:** Presiona ▶️ para la reproducción automática, o arrastra el slider para controlar manualmente.

**Qué buscar 🔍:**
1. En la superficie de **Error Cuadrático** ❌: el punto rojo puede quedar **atrapado** lejos del mínimo real.
2. En la superficie de **Pérdida Logística** ✅: el punto rojo **rueda suavemente hacia el centro** del cuenco, sin importar desde dónde empiece.

**🎯 Conclusión clave:** Cambiar de error cuadrático a pérdida logística no es solo una preferencia estilística — es la diferencia entre un problema de optimización que puede fallar y uno que **garantiza el éxito**.


In [ ]:
def plot_dual_loss_surface(learning_steps=0):
    step = int(learning_steps)
    fig  = plt.figure(figsize=(16, 7))

    configs = [
        (J_LOG_SURF, WS_LOG, BS_LOG, "log", "Perdida Logistica (NLL)  - Convexa", "viridis"),
        (J_SQ_SURF,  WS_SQ,  BS_SQ,  "sq",  "Error Cuadratico  - No Convexa", "plasma"),
    ]
    for col, (surf, ws, bs, mode, title, cmap) in enumerate(configs):
        ax = fig.add_subplot(1, 2, col + 1, projection="3d")
        ax.plot_surface(W_SURF, B_SURF, surf, cmap=cmap,
                        alpha=0.72, edgecolor="none", rstride=2, cstride=2)

        s = min(step, len(ws) - 1)
        if s > 0:
            zs = [_cost_at(ws[i], bs[i], mode) for i in range(s + 1)]
            ax.plot(ws[:s+1], bs[:s+1], zs, color=RED, lw=2, zorder=5)
            ax.scatter([ws[s]], [bs[s]], [zs[-1]], color=RED,
                       s=130, zorder=6, edgecolors="white", lw=1)
            ax.scatter([ws[0]], [bs[0]], [zs[0]], color="lime",
                       s=100, zorder=6, marker="*",
                       edgecolors="white", lw=0.8)

        ax.set_xlabel("Peso  w", fontsize=9, labelpad=8)
        ax.set_ylabel("Sesgo  b", fontsize=9, labelpad=8)
        ax.set_zlabel("Costo  J", fontsize=9, labelpad=8)
        ax.set_title(title, fontsize=11, fontweight="bold", pad=12)

    fig.suptitle(f"Comparacion de Funciones de Costo  —  Paso {step} / 55",
                 fontsize=13, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()

play2   = widgets.Play(value=0, min=0, max=55, step=1, interval=250,
                        description="Reproducir")
slider2 = widgets.IntSlider(
    value=0, min=0, max=55, step=1,
    description="Pasos GD:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="460px"),
)
widgets.jslink((play2, "value"), (slider2, "value"))
out2 = widgets.interactive_output(
    plot_dual_loss_surface, {"learning_steps": slider2})
display(widgets.VBox([
    widgets.HBox([play2, slider2]),
    out2,
]))


---
## ⛰️ Sección 3: Descenso de Gradiente — Cómo Aprende el Modelo

Tenemos un modelo y una función de pérdida. El ingrediente final es el **algoritmo de optimización** que ajusta los parámetros para minimizar el costo.

El **Descenso de Gradiente** funciona como bajar una montaña con niebla espesa 🌫️. No puedes ver el valle (el mínimo), pero puedes sentir qué dirección baja la pendiente. Das un pequeño paso en la dirección más pronunciada hacia abajo, revalúas tu posición y repites hasta llegar al fondo.

Matemáticamente, las reglas de actualización son:

$$w_j \leftarrow w_j - \alpha \frac{\partial J(\vec{w},b)}{\partial w_j}$$

$$b \leftarrow b - \alpha \frac{\partial J(\vec{w},b)}{\partial b}$$

Todos los parámetros se actualizan **simultáneamente** en cada paso.

---

### 🎚️ La Tasa de Aprendizaje $\alpha$ — Eligiendo el Tamaño de Paso

La **tasa de aprendizaje** es el hiperparámetro más crítico:

| Escenario | Comportamiento |
|---|---|
| $\alpha$ **demasiado pequeña** 🐢 | Pasos minúsculos → convergencia extremadamente lenta |
| $\alpha$ **correcta** 🚀 | Descenso suave y eficiente → convergencia rápida |
| $\alpha$ **demasiado grande** 💥 | Pasos gigantes que sobrepasan el mínimo → costo oscila o diverge |

Para la regresión logística, las derivadas parciales tienen una forma elegante e interpretable:

$$\frac{\partial J}{\partial w_j} = \frac{1}{m} \sum_{i=1}^{m} \underbrace{\left( f_{\vec{w},b}(\vec{x}^{(i)}) - y^{(i)} \right)}_{\text{error de predicción}} x_j^{(i)}$$

$$\frac{\partial J}{\partial b} = \frac{1}{m} \sum_{i=1}^{m} \left( f_{\vec{w},b}(\vec{x}^{(i)}) - y^{(i)} \right)$$

El gradiente es simplemente el **error de predicción promedio**, ponderado por la característica de entrada. Cuando el modelo ya es casi correcto (errores pequeños), el gradiente es casi cero y los parámetros apenas se mueven — el algoritmo naturalmente **desacelera al converger**.


### 🎬 Demo Interactiva 3: Viendo al Modelo Aprender en Tiempo Real

Este widget final une todo. Usamos el conjunto de datos 1D donde los pacientes con FC **por debajo de ~80 BPM** no tuvieron ataque de pánico, y los de **encima de ~80 BPM** sí.

Esta separación es intencionalmente obvia — facilita ver por qué la **curva S sigmoide es la única forma correcta** de modelar la transición abrupta mientras se mantiene acotada. ✨

**Qué hacer 👇:** Presiona ▶️ para reproducción automática, o arrastra el slider de 0 → 150 iteraciones. Observa los cuatro paneles simultáneamente.

| Panel | Qué estás observando |
|---|---|
| 📈 **Arriba izquierda** | La salida del modelo evoluciona de una línea plana a una curva S afilada centrada en 80 BPM. |
| 🗺️ **Arriba derecha** | En el espacio de parámetros, el punto rojo traza un camino hacia el mínimo global (centro más brillante). |
| 📉 **Abajo izquierda** | Curva de aprendizaje: el costo cae abruptamente al inicio, luego desacelera al converger. |
| 🎯 **Abajo derecha** | La precisión sube desde el azar hasta casi 100% mientras la frontera se ajusta alrededor de 80 BPM. |

**🤔 Pregunta para reflexionar:** ¿En cuántas iteraciones aproximadamente el modelo alcanza ~95% de precisión? ¿Por qué la precisión se estabiliza aunque el costo siga disminuyendo ligeramente?


In [ ]:
def plot_gd_matrix(iterations=0):
    t   = min(int(iterations), N_ITER)
    w_t = w_hist[t]
    b_t = b_hist[t]

    fig = plt.figure(figsize=(14, 10))
    gs  = gridspec.GridSpec(2, 2, hspace=0.40, wspace=0.33)
    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1])
    ax3 = fig.add_subplot(gs[1, 0])
    ax4 = fig.add_subplot(gs[1, 1])

    # ── Arriba izquierda: Datos + Sigmoide ───────────────────────────────
    ax1.scatter(HR1D[Y1D==0], Y1D[Y1D==0] + jitter_1d[Y1D==0],
                c=BLUE, alpha=0.65, s=55, edgecolors="white", lw=0.6,
                label="Sin ataque (y=0)", zorder=3)
    ax1.scatter(HR1D[Y1D==1], Y1D[Y1D==1] + jitter_1d[Y1D==1],
                c=RED, alpha=0.65, s=55, edgecolors="white", lw=0.6,
                label="Ataque de panico (y=1)", zorder=3)

    hr_plot   = np.linspace(50, 135, 500)
    hr_plot_s = (hr_plot - HR1D_mean) / HR1D_std
    p_hat     = sigmoid_fn(w_t * hr_plot_s + b_t)

    ax1.plot(hr_plot, p_hat, color=ORANGE, lw=3,
             label=f"Modelo (iter {t})", zorder=4)
    ax1.axhline(0.5, color="gray", lw=1.2, ls=":", alpha=0.7)
    ax1.axvline(80, color=PURPLE, lw=1.8, ls="--", alpha=0.65,
               label="Umbral 80 BPM")
    ax1.fill_between(hr_plot, 0, p_hat, alpha=0.07, color=ORANGE)
    ax1.set_xlim(50, 135)
    ax1.set_ylim(-0.18, 1.18)
    ax1.set_xlabel("Frecuencia Cardiaca (BPM)", fontweight="bold")
    ax1.set_ylabel("P(Ataque de Panico)", fontweight="bold")
    ax1.set_title("Ajuste del Modelo Sigmoide", fontweight="bold")
    ax1.legend(fontsize=9, loc="center left")

    # ── Arriba derecha: Contorno + camino GD ────────────────────────────
    j_min  = J_GRID.min()
    levels = np.linspace(j_min, j_min + 2.5, 30)
    cf2    = ax2.contourf(w_range, b_range, J_GRID,
                          levels=levels, cmap="Blues_r", alpha=0.85)
    ax2.contour(w_range, b_range, J_GRID,
                levels=levels, colors="white", linewidths=0.4, alpha=0.4)
    plt.colorbar(cf2, ax=ax2, label="Costo J(w, b)")
    if t > 0:
        ax2.plot(w_hist[:t+1], b_hist[:t+1],
                 color=RED, lw=1.8, alpha=0.85, zorder=4)
    ax2.scatter([w_hist[0]], [b_hist[0]], color="lime", s=90,
                zorder=6, marker="*", edgecolors="white", lw=0.8,
                label="Inicio")
    ax2.scatter([w_hist[t]], [b_hist[t]], color=RED, s=100,
                zorder=5, edgecolors="white", lw=1.2)
    ax2.set_xlabel("Peso  w", fontweight="bold")
    ax2.set_ylabel("Sesgo  b", fontweight="bold")
    ax2.set_title("Espacio de Parametros (Contorno)", fontweight="bold")
    ax2.legend(fontsize=9)

    # ── Abajo izquierda: Curva de aprendizaje ───────────────────────────
    iters_s = np.arange(t + 1)
    ax3.plot(iters_s, J_hist[:t+1], color=PURPLE, lw=2.5)
    ax3.scatter([t], [J_hist[t]], color=PURPLE, s=70, zorder=5)
    ax3.set_xlim(0, N_ITER)
    ax3.set_ylim(0, J_hist[0] * 1.08)
    ax3.set_xlabel("Iteracion de Descenso de Gradiente", fontweight="bold")
    ax3.set_ylabel("Costo  J(w, b)", fontweight="bold")
    ax3.set_title("Curva de Aprendizaje", fontweight="bold")
    ax3.annotate(f"J = {J_hist[t]:.4f}",
                 xy=(t, J_hist[t]),
                 xytext=(t + max(N_ITER*0.04, 5), J_hist[t] + 0.04),
                 fontsize=10, color=PURPLE,
                 arrowprops=dict(arrowstyle="->", color=PURPLE, lw=1.4))

    # ── Abajo derecha: Precision ─────────────────────────────────────────
    ax4.plot(iters_s, acc_hist[:t+1]*100, color=GREEN, lw=2.5)
    ax4.scatter([t], [acc_hist[t]*100], color=GREEN, s=70, zorder=5)
    ax4.set_xlim(0, N_ITER)
    ax4.set_ylim(40, 105)
    ax4.axhline(100, color="gray", lw=1, ls=":", alpha=0.5)
    ax4.set_xlabel("Iteracion de Descenso de Gradiente", fontweight="bold")
    ax4.set_ylabel("Precision (%)", fontweight="bold")
    ax4.set_title("Precision de Clasificacion", fontweight="bold")
    ax4.annotate(f"{acc_hist[t]*100:.1f}%",
                 xy=(t, acc_hist[t]*100),
                 xytext=(t + max(N_ITER*0.04, 5), acc_hist[t]*100 - 6),
                 fontsize=11, fontweight="bold", color=GREEN,
                 arrowprops=dict(arrowstyle="->", color=GREEN, lw=1.4))

    fig.suptitle(
        f"Descenso de Gradiente  —  Iteracion {t} / {N_ITER}"        f"   |   w = {w_t:.3f},  b = {b_t:.3f}",
        fontsize=13, fontweight="bold", y=1.01)
    plt.show()

play3   = widgets.Play(value=0, min=0, max=N_ITER, step=5, interval=150,
                        description="Reproducir")
slider3 = widgets.IntSlider(
    value=0, min=0, max=N_ITER, step=5,
    description="Iteraciones GD:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="520px"),
)
widgets.jslink((play3, "value"), (slider3, "value"))
out3 = widgets.interactive_output(plot_gd_matrix, {"iterations": slider3})
display(widgets.VBox([
    widgets.HBox([play3, slider3]),
    out3,
]))


---
## 🏆 Resumen y Conclusiones

¡Felicidades! 🎉 Has construido una comprensión completa de la Regresión Logística: desde la motivación clínica hasta ver al descenso de gradiente converger en tiempo real.

| Concepto | Idea Principal |
|---|---|
| ❌ **¿Por qué no Reg. Lineal?** | Predice valores fuera de $[0,1]$ — inválido para probabilidades. |
| ✨ **La Sigmoide** | $g(z)=\frac{1}{1+e^{-z}}$ mapea cualquier $z$ a $(0,1)$, produciendo probabilidades válidas. |
| 🔗 **Puente Log-Odds** | $e^z = \text{Odds}$ por álgebra pura; $z$ **es** el log-odds; ML y estadística hacen lo mismo. |
| 🔄 **Odds → Probabilidad** | $\hat{p} = \frac{\text{Odds}}{1+\text{Odds}} = \frac{e^z}{1+e^z} = \sigma(z)$ — cuatro formas equivalentes. |
| 💰 **MLE → Entropía Cruzada** | La función de pérdida se deriva de primeros principios, no se elige arbitrariamente. |
| 🌀 **Convexidad** | La superficie de pérdida logística tiene exactamente un mínimo — garantiza el éxito de la optimización. |
| ⛰️ **Descenso de Gradiente** | Empuja iterativamente los parámetros cuesta abajo en la superficie de costo a tasa $\alpha$. |

### 🏥 Implicaciones Clínicas

En un pipeline clínico real, las salidas de este modelo pueden usarse para:

1. 📊 **Estratificar riesgo** de pacientes antes de un procedimiento estresante.
2. 🚨 **Establecer umbrales de intervención** (e.g., administrar ansiolítico preventivo si $P(\text{pánico}) > 0.60$).
3. 🔍 **Interpretar pesos como razones de log-odds**: un aumento de una unidad en $x_j$ multiplica las odds del ataque de pánico por $e^{w_j}$.

### 📚 Lecturas Adicionales

- 🎓 **Machine Learning Specialization — Andrew Ng** ([Coursera](https://www.coursera.org/specializations/machine-learning-introduction) | [YouTube DeepLearning.AI](https://www.youtube.com/@deeplearningai)): fuente formal de la notación usada aquí.
- 📖 **"The Elements of Statistical Learning"** — Hastie, Tibshirani & Friedman. Capítulo 4.
- 📖 **"Pattern Recognition and Machine Learning"** — Bishop. Capítulo 4: fundamentos probabilísticos.
- 💻 **Documentación scikit-learn** — `LogisticRegression` y `log_loss` para uso en producción.
